In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pyproj import Proj
from sqlalchemy import create_engine, text
import psycopg2
from sklearn.model_selection import train_test_split


In [ ]:

DB_CONFIG = {
    'host': '172.31.60.8',
    'port': 5432,
    'database': 'ais',
    'user': 'postgres',
    'password': 'ANjw/84H77C9h2'
}

# Création de l'URL de connexion
DATABASE_URL = f"postgresql://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Création du moteur SQLAlchemy
engine = create_engine(DATABASE_URL)

# Configuration des splits (70% train, 15% val, 15% test)
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print(f"Configuration DB chargée")
print(f"   - Host: {DB_CONFIG['host']}")
print(f"   - Database: {DB_CONFIG['database']}")
print(f"   - Splits: Train={TRAIN_RATIO:.0%}, Val={VAL_RATIO:.0%}, Test={TEST_RATIO:.0%}")


In [ ]:
def check_data_volume():
    """Vérifie le volume de données disponibles dans la base"""
    
    query = """
        SELECT 
            COUNT(*) as total_points,
            COUNT(DISTINCT v.mmsi) as unique_vessels,
            MIN(pv.base_date_time) as earliest_date,
            MAX(pv.base_date_time) as latest_date,
            EXTRACT(EPOCH FROM (MAX(pv.base_date_time) - MIN(pv.base_date_time))) / 86400 as duration_days
        FROM position_vessel pv
        JOIN vessel v ON pv.vessel_id = v.id
        WHERE pv.sog IS NOT NULL AND pv.cog IS NOT NULL
    """
    
    try:
        df = pd.read_sql(query, engine)
        
        print("="*60)
        print("VOLUME DE DONNEES BRUTES DISPONIBLES")
        print("="*60)
        print(f"Points totaux      : {df['total_points'].iloc[0]:,}")
        print(f"Navires uniques    : {df['unique_vessels'].iloc[0]:,}")
        print(f"Periode            : {df['earliest_date'].iloc[0]} -> {df['latest_date'].iloc[0]}")
        print(f"Duree              : {df['duration_days'].iloc[0]:.1f} jours")
        
        # Estimation du nombre de trajectoires
        points = df['total_points'].iloc[0]
        vessels = df['unique_vessels'].iloc[0]
        avg_points_per_vessel = points / vessels if vessels > 0 else 0
        estimated_trajs = int(avg_points_per_vessel / 100)
        
        print("\nESTIMATION APRES PRETRAITEMENT")
        print(f"   Points moyens/navire : {avg_points_per_vessel:.0f}")
        print(f"   Trajectoires estimees: ~{estimated_trajs * vessels}")
        
        # Recommandation
        print("\nRECOMMANDATION")
        if points < 10000:
            print("   INSUFFISANT - Besoin minimum 10,000 points")
            print("   Collectez plus de donnees avant l'entrainement")
        elif points < 50000:
            print("   MINIMAL - Faisable mais resultats limites")
            print("   Entrainement possible, mais precision reduite")
        elif points < 500000:
            print("   BON - Volume suffisant pour l'entrainement")
            print("   Resultats attendus corrects")
        else:
            print("   EXCELLENT - Volume optimal")
            print("   Excellents resultats attendus")
        
        print("="*60)
        return df
    
    except Exception as e:
        print(f"Erreur lors de la verification: {e}")
        return None

# Lancer le diagnostic
check_data_volume()


In [ ]:

def load_raw_data_from_db(limit=None):
    """Charge les données brutes depuis PostgreSQL (position_vessel + vessel)"""
    query = """
        SELECT 
            v.mmsi as "MMSI",
            pv.base_date_time as "BaseDateTime",
            ST_Y(pv.geom::geometry) as "LAT",
            ST_X(pv.geom::geometry) as "LON",
            pv.sog as "SOG",
            pv.cog as "COG",
            v.vessel_type_id as "VesselType"
        FROM position_vessel pv
        JOIN vessel v ON pv.vessel_id = v.id
        WHERE pv.sog IS NOT NULL AND pv.cog IS NOT NULL
        ORDER BY v.mmsi, pv.base_date_time
    """
    if limit:
        query = query.replace("ORDER BY", f"LIMIT {limit} ORDER BY")
    
    print(f"Chargement des donnees depuis position_vessel...")
    df = pd.read_sql(query, engine)
    print(f"   {len(df):,} lignes chargees")
    return df


def save_processed_to_db(trajectories, splits_assignment):
    """Sauvegarde les trajectoires prétraitées dans PostgreSQL"""
    print(f"Sauvegarde des trajectoires pretraitees...")
    
    all_points = []
    for traj_id, traj in enumerate(trajectories):
        split = splits_assignment[traj_id]
        mmsi = str(traj["MMSI"].iloc[0]) if "MMSI" in traj.columns else "unknown"
        
        for _, row in traj.iterrows():
            all_points.append({
                'trajectory_id': traj_id,
                'mmsi': mmsi,
                'base_date_time': row['BaseDateTime'],
                'lat': row['LAT'],
                'lon': row['LON'],
                'sog': row['SOG'],
                'cog': row['COG'],
                'dataset_split': split
            })
    
    df_points = pd.DataFrame(all_points)
    
    # Vider et remplir la table
    with engine.connect() as conn:
        conn.execute(text("TRUNCATE TABLE trajectory_processed"))
        conn.commit()
    
    df_points.to_sql('trajectory_processed', engine, if_exists='append', 
                     index=False, method='multi', chunksize=10000)
    
    # Afficher les stats
    stats = df_points.groupby('dataset_split').agg(
        trajectoires=('trajectory_id', 'nunique'),
        points=('trajectory_id', 'count')
    )
    print(f"   {len(df_points):,} points sauvegardes")
    print(stats)


def load_processed_from_db(split=None):
    """Charge les données prétraitées (optionnel: par split)"""
    query = 'SELECT * FROM trajectory_processed'
    if split:
        query += f" WHERE dataset_split = '{split}'"
    query += ' ORDER BY trajectory_id, base_date_time'
    
    print(f"Chargement (split={split or 'all'})...")
    df = pd.read_sql(query, engine)
    df.columns = ['id', 'trajectory_id', 'MMSI', 'BaseDateTime', 'LAT', 'LON', 'SOG', 'COG', 'dataset_split']
    print(f"   {len(df):,} lignes")
    return df


def check_table_exists(table_name):
    """Vérifie si une table existe"""
    query = text("SELECT EXISTS (SELECT FROM information_schema.tables WHERE table_name = :t)")
    with engine.connect() as conn:
        return conn.execute(query, {"t": table_name}).scalar()


def get_split_stats():
    """Statistiques des splits"""
    query = """
        SELECT dataset_split, 
               COUNT(DISTINCT trajectory_id) as trajectoires,
               COUNT(*) as points
        FROM trajectory_processed 
        GROUP BY dataset_split
    """
    return pd.read_sql(query, engine)


print("Fonctions DB chargees")


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_IN = 20
BATCH_SIZE = 64
EPOCHS = 40
LR = 1e-4

amax_knots = 0.15
amax = amax_knots * 0.514
proj = Proj(proj="utm", zone=31, ellps="WGS84")


In [3]:
def clean_ais(df):
    df = df.copy()

    df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
    df = df.sort_values(["MMSI", "BaseDateTime"])
    df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
    df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
    df['SOG'] = pd.to_numeric(df['SOG'], errors='coerce')
    df['COG'] = pd.to_numeric(df['COG'], errors='coerce')
    
    df = df.drop_duplicates(subset=["MMSI", "BaseDateTime"])
    df = df[df["SOG"] > 0]
    df = df[df["SOG"] <= 30]
    df = df[df["COG"].between(0, 360)]
    df = df[df["VesselType"].between(20, 90)]

    df["dt"] = df.groupby("MMSI")["BaseDateTime"].diff().dt.total_seconds()
    df = df[(df["dt"].isna()) | (df["dt"] >= 1)]

    return df.drop(columns=["dt"])


In [4]:
def extract_trajectories(df):
    trajs = []

    for mmsi, g in df.groupby("MMSI"):
        g = g.sort_values("BaseDateTime").copy()
        g["dt"] = g["BaseDateTime"].diff().dt.total_seconds()
        g["traj_id"] = (g["dt"] > 600).cumsum()

        for _, t in g.groupby("traj_id"):
            if len(t) < 24:
                continue
            duration = (t["BaseDateTime"].iloc[-1] - t["BaseDateTime"].iloc[0]).total_seconds() / 3600
            if duration > 24:
                continue
            if t["SOG"].mean() < 1:
                continue

            trajs.append(t.reset_index(drop=True))

    return trajs


In [5]:
def correct_speed(traj):
    sog = traj["SOG"].values.copy()
    t = traj["BaseDateTime"].values

    for i in range(1, len(traj)-1):
        dt1 = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
        if dt1 <= 0 or dt2 <= 0:
            continue

        if abs(sog[i] - sog[i-1]) > amax_knots * dt1:
            sog[i] = (dt1*sog[i-1] + dt2*sog[i+1]) / (dt1 + dt2)

    traj["SOG"] = sog
    return traj


In [6]:
def correct_position(traj):
    lat = traj["LAT"].values
    lon = traj["LON"].values
    t = traj["BaseDateTime"].values
    sog = traj["SOG"].values

    x, y = proj(lon, lat)

    for i in range(1, len(traj)-1):
        dt = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        if dt <= 0:
            continue

        v = sog[i-1] * 0.514
        dmax = v*dt + 0.5*amax*(dt**2)
        d = math.hypot(x[i]-x[i-1], y[i]-y[i-1])

        if d > dmax:
            dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
            if dt2 <= 0:
                continue

            v2 = sog[i] * 0.514
            dmax2 = v2*dt2 + 0.5*amax*(dt2**2)
            d2 = math.hypot(x[i+1]-x[i], y[i+1]-y[i])

            if d2 > dmax2:
                alpha = (t[i]-t[i-1]) / (t[i+1]-t[i-1])
                x[i] = x[i-1] + alpha*(x[i+1]-x[i-1])
                y[i] = y[i-1] + alpha*(y[i+1]-y[i-1])

    lon_c, lat_c = proj(x, y, inverse=True)
    traj["LAT"] = lat_c
    traj["LON"] = lon_c
    return traj


In [7]:
def resample(traj):
    for col in ["LAT", "LON", "SOG", "COG"]:
        traj[col] = pd.to_numeric(traj[col], errors='coerce')

    # Resample et interpolation
    traj = traj.set_index("BaseDateTime").resample("1min").interpolate()
    x, y = proj(traj["LON"].values, traj["LAT"].values)
    dist = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    traj["SOG"] = np.insert(dist/60/0.514, 0, traj["SOG"].iloc[0])

    cog = np.unwrap(np.deg2rad(traj["COG"].values))
    traj["COG"] = np.rad2deg(cog) % 360

    return traj.reset_index()


In [8]:
def plot_before_after(raw_trajs, corr_trajs, n=5):
    plt.figure(figsize=(10,8))

    for i in range(n):
        plt.plot(raw_trajs[i]["LON"], raw_trajs[i]["LAT"], 'r--', alpha=0.5)

    for i in range(n):
        plt.plot(corr_trajs[i]["LON"], corr_trajs[i]["LAT"], 'b')

    plt.legend(["Raw", "Corrected"])
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Trajectories Before vs After Correction")
    plt.grid(True)
    plt.show()


In [9]:
class AISTrajDataset(Dataset):
    def __init__(self, df):
        self.X, self.Y = [], []

        for _, g in df.groupby("MMSI"):
            lat, lon = g["LAT"].values, g["LON"].values
            sog = g["SOG"].values / 30
            cog = np.deg2rad(g["COG"].values)

            dx = (lat - lat[0]) * 111320
            dy = (lon - lon[0]) * 111320 * np.cos(np.deg2rad(lat[0]))

            feats = np.stack([dx, dy, sog, np.sin(cog), np.cos(cog)], axis=1)

            for i in range(len(feats)-SEQ_IN):
                self.X.append(feats[i:i+SEQ_IN])
                self.Y.append(feats[i+1:i+SEQ_IN+1])

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]


In [10]:
class LSTMTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(5, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        enc = nn.TransformerEncoderLayer(128, 8, batch_first=True)
        self.tr = nn.TransformerEncoder(enc, 2)
        self.out = nn.Linear(128, 5)

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.lstm(x)
        x = self.tr(x)
        return self.out(x)


In [11]:
def train(model, loader):
    opt = torch.optim.Adam(model.parameters(), LR)
    loss_fn = nn.MSELoss()
    model.train()

    for e in range(EPOCHS):
        tot = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"Epoch {e+1} | MSE {tot/len(loader.dataset):.6f}")


In [12]:
def mse(pred, true):
    return ((pred - true) ** 2).mean()


def rmse(pred, true):
    return torch.sqrt(mse(pred, true))


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = torch.deg2rad(lat1), torch.deg2rad(lat2)
    dphi = phi2 - phi1
    dl = torch.deg2rad(lon2 - lon1)
    a = torch.sin(dphi/2)**2 + torch.cos(phi1)*torch.cos(phi2)*torch.sin(dl/2)**2
    return 2 * R * torch.atan2(torch.sqrt(a), torch.sqrt(1-a))


def ade_fde(pred, true):
    ade = haversine(
        pred[:, :, 0], pred[:, :, 1],
        true[:, :, 0], true[:, :, 1]
    ).mean()

    fde = haversine(
        pred[:, -1, 0], pred[:, -1, 1],
        true[:, -1, 0], true[:, -1, 1]
    ).mean()

    return ade.item(), fde.item()

In [ ]:
def evaluate(model, loader):
    """Évalue le modèle et retourne les métriques"""
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            pred = model(x).cpu()
            preds.append(pred)
            trues.append(y)

    preds = torch.cat(preds)
    trues = torch.cat(trues)

    mse_v = mse(preds, trues).item()
    rmse_v = rmse(preds, trues).item()
    ade, fde = ade_fde(preds, trues)

    print(f"MSE  : {mse_v:.6f}")
    print(f"RMSE : {rmse_v:.4f}")
    print(f"ADE  : {ade:.2f} m")
    print(f"FDE  : {fde:.2f} m")
    
    return {
        'mse': mse_v,
        'rmse': rmse_v,
        'ade': ade,
        'fde': fde
    }


In [ ]:
# =============================================================================
# ÉTAPE 1 : PRÉTRAITEMENT ET SAUVEGARDE EN DB
# =============================================================================

def assign_splits(n_trajectories, random_state=42):
    """Assigne chaque trajectoire à train/val/test"""
    indices = np.arange(n_trajectories)
    train_idx, temp_idx = train_test_split(indices, train_size=TRAIN_RATIO, random_state=random_state)
    val_ratio_adj = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
    val_idx, test_idx = train_test_split(temp_idx, train_size=val_ratio_adj, random_state=random_state)
    
    splits = {}
    for i in train_idx: splits[i] = 'train'
    for i in val_idx: splits[i] = 'val'
    for i in test_idx: splits[i] = 'test'
    return splits


def run_preprocessing():
    """Pipeline de prétraitement complet"""
    # 1. Charger données brutes
    df = load_raw_data_from_db()
    if len(df) == 0:
        print("Aucune donnee!")
        return None
    
    # 2. Nettoyage
    print("\nNettoyage...")
    df = clean_ais(df)
    print(f"   {len(df):,} lignes")
    
    # 3. Extraction trajectoires
    print("\nExtraction trajectoires...")
    raw_trajs = extract_trajectories(df)
    print(f"   {len(raw_trajs)} trajectoires")
    
    if len(raw_trajs) == 0:
        print("Aucune trajectoire!")
        return None
    
    # 4. Correction et rééchantillonnage
    print("\nCorrection...")
    corr_trajs = []
    for i, t in enumerate(raw_trajs):
        for _ in range(3):
            t = correct_speed(t)
            t = correct_position(t)
        corr_trajs.append(resample(t))
        if (i + 1) % 100 == 0:
            print(f"   {i + 1}/{len(raw_trajs)}")
    
    # 5. Visualisation
    plot_before_after(raw_trajs, corr_trajs, n=min(5, len(raw_trajs)))
    
    # 6. Splits
    print("\nAssignation splits...")
    splits = assign_splits(len(corr_trajs))
    
    # 7. Sauvegarde
    save_processed_to_db(corr_trajs, splits)
    
    print("\nPRETRAITEMENT TERMINE")
    return corr_trajs


# Vérifier si données existent
if check_table_exists('trajectory_processed'):
    try:
        stats = get_split_stats()
        if len(stats) > 0:
            print("Donnees pretraitees presentes:")
            print(stats)
            print("\nPour recreer: run_preprocessing()")
        else:
            print("Table vide, lancement pretraitement...")
            run_preprocessing()
    except:
        print("Lancement pretraitement...")
        run_preprocessing()
else:
    print("Table trajectory_processed non trouvee!")
    print("   Executez d'abord la migration V10")


In [ ]:
# =============================================================================
# ÉTAPE 2 : ENTRAÎNEMENT DU MODÈLE
# =============================================================================

def train_with_validation(model, train_loader, val_loader):
    """Entraînement avec early stopping"""
    opt = torch.optim.Adam(model.parameters(), LR)
    loss_fn = nn.MSELoss()
    best_val_loss = float('inf')
    patience, patience_counter = 5, 0
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss = sum(
            (opt.zero_grad(), loss_fn(model(x.to(DEVICE)), y.to(DEVICE)).backward(), opt.step(), 
             loss_fn(model(x.to(DEVICE)), y.to(DEVICE)).item() * x.size(0))[3]
            for x, y in train_loader
        ) / len(train_loader.dataset)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_loss = sum(
                loss_fn(model(x.to(DEVICE)), y.to(DEVICE)).item() * x.size(0)
                for x, y in val_loader
            ) / len(val_loader.dataset)
        
        print(f"Epoch {epoch+1:2d}/{EPOCHS} | Train: {train_loss:.6f} | Val: {val_loss:.6f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping epoch {epoch+1}")
                model.load_state_dict(torch.load("best_model.pth"))
                break


def run_training():
    """Pipeline d'entraînement complet"""
    print("="*50)
    print("ENTRAINEMENT")
    print("="*50)
    
    # Charger les données
    df_train = load_processed_from_db('train')
    df_val = load_processed_from_db('val')
    df_test = load_processed_from_db('test')
    
    # Créer datasets
    print("\nCreation datasets...")
    train_ds = AISTrajDataset(df_train)
    val_ds = AISTrajDataset(df_val)
    test_ds = AISTrajDataset(df_test)
    print(f"   Train: {len(train_ds):,} | Val: {len(val_ds):,} | Test: {len(test_ds):,}")
    
    # DataLoaders
    train_loader = DataLoader(train_ds, BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, BATCH_SIZE)
    test_loader = DataLoader(test_ds, BATCH_SIZE)
    
    # Modèle
    print(f"\nModele sur {DEVICE}")
    model = LSTMTransformer().to(DEVICE)
    
    # Entraînement
    print("\nEntrainement...")
    train_with_validation(model, train_loader, val_loader)
    
    # Évaluation
    print("\nEvaluation Test Set:")
    metrics = evaluate(model, test_loader)
    
    # Sauvegarde
    model_path = f"ais_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"\nModele: {model_path}")
    
    return model, metrics


In [ ]:

try:
    stats = get_split_stats()
    if len(stats) > 0:
        print("Donnees disponibles:")
        print(stats)
        print("\n")
        model, metrics = run_training()
        print("\nTERMINE")
    else:
        print("Pas de donnees! Executez: run_preprocessing()")
except Exception as e:
    print(f"Erreur: {e}")
    print("   Verifiez la connexion DB et la migration V10")
